In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import io

In [ ]:
log_file = 'webruns/round2_scrape.log'

In [ ]:
import json
import pandas as pd
import io

with open(log_file, 'r') as file:
    file_content = file.read()
    sections = file_content.split('Sandbox logs:')[1].split('Activities log:')
    
    # Parse the Sandbox logs section into a JSON array
    sandbox_logs = []
    logs_data = sections[0].strip()
    
    # Split the logs data into individual JSON objects
    start_index = 0
    while start_index < len(logs_data):
        if logs_data[start_index] == '{':
            end_index = logs_data.find('}', start_index) + 1
            log_entry = logs_data[start_index:end_index]
            sandbox_logs.append(json.loads(log_entry))
            start_index = end_index
        else:
            start_index += 1
    
    # Create a DataFrame from the parsed JSON data
    df_sandbox = pd.DataFrame(sandbox_logs)
    
    # Extract the ORCHIDS POSITION value from the lambdaLog column
    df_sandbox['orchids_position'] = df_sandbox['lambdaLog'].str.extract(r'ORCHIDS POSITION: (\-?\d+)').astype(int)
    
    # Select the desired columns for the final DataFrame
    df_sandbox = df_sandbox[['timestamp', 'orchids_position']]
    
    activities_log = sections[1].split('Trade History:')[0]
    df_activities = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    
    trade_json = pd.json_normalize(json.loads(sections[1].split('Trade History:')[1])).to_json()


In [ ]:
df_sandbox

In [ ]:
import json
import pandas as pd
import io

with open(log_file, 'r') as file:
    file_content = file.read()
    sections = file_content.split('Sandbox logs:')[1].split('Activities log:')
    
    # Parse the Sandbox logs section into a JSON array
    sandbox_logs = []
    logs_data = sections[0].strip()
    
    # Split the logs data into individual JSON objects
    start_index = 0
    while start_index < len(logs_data):
        if logs_data[start_index] == '{':
            end_index = logs_data.find('}', start_index) + 1
            log_entry = logs_data[start_index:end_index]
            sandbox_logs.append(json.loads(log_entry))
            start_index = end_index
        else:
            start_index += 1
    
    # Create a DataFrame from the parsed JSON data
    df_sandbox = pd.DataFrame(sandbox_logs)
    
    # Extract the ORCHIDS POSITION value from the lambdaLog column
    df_sandbox['orchids_position'] = df_sandbox['lambdaLog'].str.extract(r'ORCHIDS POSITION: (\-?\d+)').astype(int)
    
    # Extract the IMPLIED_BID value from the lambdaLog column
    df_sandbox['implied_bid'] = df_sandbox['lambdaLog'].str.extract(r'IMPLIED_BID: (\d+\.\d+)').astype(float)
    
    # Extract the IMPLIED_ASK value from the lambdaLog column
    df_sandbox['implied_ask'] = df_sandbox['lambdaLog'].str.extract(r'IMPLIED_ASK: (\d+\.\d+)').astype(float)
    
    # Extract the CURR EDGE value from the lambdaLog column
    df_sandbox['curr_edge'] = df_sandbox['lambdaLog'].str.extract(r'CURR EDGE: (\d+)').astype(int)
    
    # Select the desired columns for the final DataFrame
    df_sandbox = df_sandbox[['timestamp', 'orchids_position', 'implied_bid', 'implied_ask', 'curr_edge']]
    
    activities_log = sections[1].split('Trade History:')[0]
    df_activities = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    
    trade_json = pd.json_normalize(json.loads(sections[1].split('Trade History:')[1])).to_json()

# Display the DataFrame
display(df_sandbox)

In [ ]:
df_sandbox.isna().mean()

In [ ]:
def parse_log_file(log_file, product):
    with open(log_file, "r") as file:
        file_content = file.read()
    sections = file_content.split("Sandbox logs:")[1].split("Activities log:")
    activities_log = sections[1].split("Trade History:")[0]
    df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_json = pd.json_normalize(
        json.loads(sections[1].split("Trade History:")[1])
    ).to_json()
    sandbox_logs = []
    logs_data = sections[0].strip()
    start_index = 0
    while start_index < len(logs_data):
        if logs_data[start_index] == "{":
            end_index = logs_data.find("}", start_index) + 1
            log_entry = logs_data[start_index:end_index]
            sandbox_logs.append(json.loads(log_entry))
            start_index = end_index
        else:
            start_index += 1
    df_sandbox = pd.DataFrame(sandbox_logs)
    if "lambdaLog" in df_sandbox.columns:
        if df_sandbox["lambdaLog"].str.contains("IMPLIED_BID").any():
            implied_bid = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"IMPLIED_BID: (\d+\.\d+)")[0], errors='coerce')
            implied_bid_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "implied_bid": implied_bid, "product": "ORCHIDS"})
            df = pd.merge(df, implied_bid_df, on=["timestamp", "product"], how="left")
        if df_sandbox["lambdaLog"].str.contains("IMPLIED_ASK").any():
            implied_ask = pd.to_numeric(df_sandbox["lambdaLog"].str.extract(r"IMPLIED_ASK: (\d+\.\d+)")[0], errors='coerce')
            implied_ask_df = pd.DataFrame({"timestamp": df_sandbox["timestamp"], "implied_ask": implied_ask, "product": "ORCHIDS"})
            df = pd.merge(df, implied_ask_df, on=["timestamp", "product"], how="left")
    return df, trade_json, df_sandbox.to_json()


In [ ]:
df = parse_log_file(log_file, "ORCHIDS")[0]

In [ ]:
df

In [ ]:
df_orchids = df[df['product'] == 'ORCHIDS']

In [ ]:
df_orchids